Predicting NFL Rushing Yards from Player Tracking Data

Practice project on the closed 2020 NFL Big Data Bowl (Kaggle), using single-frame player tracking data at the moment of handoff to predict rushing yards.

The question I actually needed to answer first: what does "predicting well" mean here? Early hand-built models scored ~3.7 MAE and I couldn't tell if that was good. The fix was switching to the competition's own metric (CRPS, a score over the full probability distribution of yardage) and benchmarking against a floor — a model that ignores the players entirely and just guesses the historical yardage distribution every time. That floor scores 0.01427. Anything that can't beat that isn't using the tracking data for anything.

Hand-built features (exploration notebook): distance-based and closing-speed features (nearest defender, linebacker/safety distances, an interception-margin model, a "defenders near the offensive line and closing" feature) were tested one at a time under cross-validation. Most were null. One survived — a feature summing the closing speed of defenders within ~3 yards of an offensive lineman — giving a modest, consistently positive gain across game-grouped and season-holdout splits. Best hand-built result: +2.4% CRPS skill over the floor (R² ≈ 0.03 on raw yards).

Along the way, I found that PlayerAccel looked like the dominant feature (permutation importance 2x the next-highest) but the effect was mostly a data artifact: 2017 rows have a much stronger (and physically implausible) correlation between acceleration and yards than 2018-19, consistent with known timing/measurement issues in that season. Scoring only on 2018+ rows dropped the "real" baseline R² from ~0.05 to ~0.024.

A small version of the winning approach: the 2020 competition's 1st-place solution ("The Zoo," Singer & Gordeev) used a CNN over defender-vs-blocker relative position/velocity pairs, with no hand-engineered features. I built a small version of that architecture (credit above; this is my own implementation from their published description, not their code) — same 5 relative vectors, pooling over players, CRPS loss, left-right mirror augmentation. One untuned run, 15 epochs, no bagging:

+12.85% CRPS skill over the floor, R² ≈ 0.13 on raw yards — about 5x the hand-built model, roughly 85% of the way to the published winning score (~+15%) with none of their tuning or ensembling.

Takeaway: on this dataset, the useful signal lives in relative player-pair geometry, not in aggregate handoff-frame stats. That's consistent with why most of my hand-built features (which summarized distances and speeds into single numbers per play) came back null — they were throwing away exactly the pairwise structure the network exploits.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/nfl-big-data-bowl-2020/train.csv
/kaggle/input/competitions/nfl-big-data-bowl-2020/kaggle/competitions/nflrush/test.csv.encrypted
/kaggle/input/competitions/nfl-big-data-bowl-2020/kaggle/competitions/nflrush/sample_submission.csv.encrypted
/kaggle/input/competitions/nfl-big-data-bowl-2020/kaggle/competitions/nflrush/__init__.py
/kaggle/input/competitions/nfl-big-data-bowl-2020/kaggle/competitions/nflrush/competition.cpython-36m-x86_64-linux-gnu.so


In [2]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold

df = pd.read_csv('/kaggle/input/competitions/nfl-big-data-bowl-2020/train.csv', low_memory=False)
print(df.shape)   # (682154, 49)

(682154, 49)


In [3]:
df['ToLeft'] = df['PlayDirection'] == 'left'
df['X_std'] = df['X']
df.loc[df['ToLeft'], 'X_std'] = 120 - df.loc[df['ToLeft'], 'X']
df['Y_std'] = df['Y']
df.loc[df['ToLeft'], 'Y_std'] = (160/3) - df.loc[df['ToLeft'], 'Y']
df['Dir_std'] = df['Dir']
df.loc[df['ToLeft'], 'Dir_std'] = (df.loc[df['ToLeft'], 'Dir'] + 180) % 360

In [4]:
rush = (df.loc[df['NflId'] == df['NflIdRusher'], ['PlayId', 'X_std', 'Y_std', 'Team']]
          .rename(columns={'X_std': 'rX', 'Y_std': 'rY', 'Team': 'rTeam'}))

meta = df.drop_duplicates('PlayId').set_index('PlayId')[['GameId', 'Season', 'Yards']]

yards_grid = np.arange(-99, 100)
def crps(cdf_pred, y):
    truth = (yards_grid[None, :] >= y[:, None]).astype(float)
    return ((cdf_pred - truth) ** 2).mean()

In [5]:
use = df[['PlayId', 'NflId', 'NflIdRusher', 'Team', 'X_std', 'Y_std', 'Dir_std', 'Dis']].merge(
    rush[['PlayId', 'rTeam']], on='PlayId')

sp = use['Dis'] * 10                     # cleaned speed (Dis*10), per The Zoo
ang = np.radians(90 - use['Dir_std'])
use['vx'] = sp * np.cos(ang)
use['vy'] = sp * np.sin(ang)
use['role'] = np.where(use['NflId'] == use['NflIdRusher'], 'rusher',
                np.where(use['Team'] != use['rTeam'], 'def', 'off'))

feat = ['X_std', 'Y_std', 'vx', 'vy']
tensors, pids, dropped = [], [], 0

for pid, g in use.sort_values(['PlayId', 'NflId']).groupby('PlayId'):
    r, d, o = g[g.role == 'rusher'], g[g.role == 'def'], g[g.role == 'off']
    if len(r) != 1 or len(d) != 11 or len(o) != 10:
        dropped += 1
        continue
    R, D, O = r[feat].values[0], d[feat].values, o[feat].values
    if np.isnan(R).any() or np.isnan(D).any() or np.isnan(O).any():
        dropped += 1
        continue

    def_vel   = np.broadcast_to(D[:, None, 2:4], (11, 10, 2))                    # ch 0-1
    def_pos_r = np.broadcast_to((D[:, 0:2] - R[0:2])[:, None, :], (11, 10, 2))   # ch 2-3
    def_vel_r = np.broadcast_to((D[:, 2:4] - R[2:4])[:, None, :], (11, 10, 2))   # ch 4-5
    off_pos_d = O[None, :, 0:2] - D[:, None, 0:2]                                # ch 6-7
    off_vel_d = O[None, :, 2:4] - D[:, None, 2:4]                                # ch 8-9

    T = np.concatenate([def_vel, def_pos_r, def_vel_r, off_pos_d, off_vel_d], axis=2)   # (11, 10, 10)
    tensors.append(T.transpose(2, 0, 1))                                                  # (10, 11, 10)
    pids.append(pid)

X_t = np.stack(tensors).astype(np.float32)
pids = np.array(pids)
print("tensor shape:", X_t.shape, "| plays dropped:", dropped, "| NaNs:", np.isnan(X_t).sum())
print("mean (defender X - rusher X):", X_t[:, 2].mean().round(2), "-> expect positive")
print("mean (offense X - defender X):", X_t[:, 6].mean().round(2), "-> expect negative")
print("mean (defender vx):", X_t[:, 0].mean().round(2), "-> expect near or below zero")

tensor shape: (30987, 10, 11, 10) | plays dropped: 20 | NaNs: 0
mean (defender X - rusher X): 8.01 -> expect positive
mean (offense X - defender X): -3.9 -> expect negative
mean (defender vx): -0.41 -> expect near or below zero


In [6]:
y_all = meta.loc[pids, 'Yards'].values
g_all = meta.loc[pids, 'GameId'].values
s_all = meta.loc[pids, 'Season'].values
folds = list(GroupKFold(5).split(X_t, groups=g_all))

fl = []
for tr, te in folds:
    te = te[s_all[te] >= 2018]
    cdf = (np.clip(y_all[tr], -99, 99)[:, None] <= yards_grid[None, :]).mean(axis=0)
    fl.append(crps(np.tile(cdf, (len(te), 1)), y_all[te]))
print("floor:", np.round(fl, 5), "mean", round(np.mean(fl), 5))

floor: [0.01427 0.01436 0.01447 0.01419 0.01406] mean 0.01427


In [7]:
import pickle
with open('/kaggle/working/folds.pkl', 'wb') as f:
    pickle.dump({'pids': pids, 'y_all': y_all, 'g_all': g_all, 's_all': s_all,
                 'folds': folds, 'fl': fl}, f)
print("pinned:", len(folds), "folds, floor mean", round(np.mean(fl), 5))

pinned: 5 folds, floor mean 0.01427


In [8]:
# Reload check — run this instead of recomputing folds if resuming a fresh session
with open('/kaggle/working/folds.pkl', 'rb') as f:
    saved = pickle.load(f)
assert np.array_equal(pids, saved['pids']), "tensor plays don't match the pinned split — rebuild folds.pkl"
folds, fl = saved['folds'], saved['fl']
print("reloaded pinned floor mean:", round(np.mean(fl), 5))

reloaded pinned floor mean: 0.01427


In [9]:
import torch, torch.nn as nn, torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

Y_CH = [1, 3, 5, 7, 9]   # channels holding y-components; mirroring the field negates these

def pool(x, dim, w_avg=0.7):     # weighted mix of average and max pooling, as in the write-up
    return w_avg * x.mean(dim=dim) + (1 - w_avg) * x.max(dim=dim).values

class ZooLite(nn.Module):
    def __init__(self, c_in=10, h=96):
        super().__init__()
        self.pair = nn.Sequential(                       # learns from defender-offense pairs
            nn.Conv2d(c_in, h, 1), nn.ReLU(),
            nn.Conv2d(h, h, 1), nn.ReLU(),
            nn.Conv2d(h, h, 1), nn.ReLU())
        self.defn = nn.Sequential(                       # learns per defender, after pooling over offense
            nn.Conv1d(h, h, 1), nn.ReLU(), nn.BatchNorm1d(h),
            nn.Conv1d(h, h, 1), nn.ReLU(), nn.BatchNorm1d(h),
            nn.Conv1d(h, h, 1), nn.ReLU(), nn.BatchNorm1d(h))
        self.head = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.LayerNorm(128),
            nn.Linear(128, 199))                          # one logit per yard, -99..99

    def forward(self, x):                                 # x: (B, 10, 11, 10)
        x = self.pair(x)                                  # (B, h, 11, 10)
        x = pool(x, dim=3)                                # pool over offense -> (B, h, 11)
        x = self.defn(x)                                  # (B, h, 11)
        x = pool(x, dim=2)                                # pool over defenders -> (B, h)
        return torch.cumsum(F.softmax(self.head(x), dim=1), dim=1)   # predicted CDF, (B, 199)

grid_t = torch.tensor(yards_grid, dtype=torch.float32, device=device)
def crps_loss(cdf, y):
    truth = (grid_t[None, :] >= y[:, None]).float()
    return ((cdf - truth) ** 2).mean()

device: cuda


In [10]:
m = ZooLite().to(device)
xb = torch.tensor(X_t[:8] / 5.0).to(device)
out = m(xb)
print("output shape:", tuple(out.shape), "(expect (8, 199))")
print("CDF increasing?", bool((out[:, 1:] >= out[:, :-1] - 1e-6).all()), "| last value ~1:", out[:, -1].round(decimals=3).tolist())
print("params:", sum(p.numel() for p in m.parameters()))

output shape: (8, 199) (expect (8, 199))
CDF increasing? True | last value ~1: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
params: 86535


In [11]:
SCALE = 5.0                                   # fixed constant so inputs aren't huge; not fit to the data
Xs = torch.tensor(X_t / SCALE)
y_train_all = np.clip(y_all, -30, 50).astype(np.float32)   # clipped targets for training, as in the write-up

def train_fold(tr, epochs=15, bs=128, lr_hi=2e-3, seed=0):
    torch.manual_seed(seed)
    model = ZooLite().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr_hi / 2)
    steps = epochs * int(np.ceil(len(tr) / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr_hi, total_steps=steps)
    Xtr = Xs[torch.as_tensor(tr)]
    ytr = torch.tensor(y_train_all[tr])
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr))
        tot = 0.0
        for i in range(0, len(tr), bs):
            b = perm[i:i + bs]
            xb, yb = Xtr[b].to(device), ytr[b].to(device)
            flip = (torch.rand(len(b), device=device) < 0.5).view(-1, 1, 1, 1)   # 50% mirror augmentation
            xb[:, Y_CH] = torch.where(flip, -xb[:, Y_CH], xb[:, Y_CH])
            loss = crps_loss(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * len(b)
        if ep % 5 == 0 or ep == epochs - 1:
            print(f"  epoch {ep+1:2d}/{epochs}  train CRPS {tot/len(tr):.5f}")
    return model

def predict_cdf(model, idx):
    model.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(idx), 512):
            xb = Xs[torch.as_tensor(idx[i:i + 512])].to(device)
            xf = xb.clone(); xf[:, Y_CH] *= -1                                   # mirrored copy for test-time averaging
            outs.append((0.5 * (model(xb) + model(xf))).cpu().numpy())
    return np.concatenate(outs)

def run_fold(k, epochs=15):
    tr, te = folds[k]
    te = te[s_all[te] >= 2018]                                                   # score on 2018+ rows only
    model = train_fold(tr, epochs)
    return crps(predict_cdf(model, te), y_all[te].astype(float))

score0 = run_fold(0)
print(f"fold 0 CRPS {score0:.5f} | floor {fl[0]:.5f} | skill vs floor {1 - score0/fl[0]:+.2%}")

  epoch  1/15  train CRPS 0.03537
  epoch  6/15  train CRPS 0.01243
  epoch 11/15  train CRPS 0.01202
  epoch 15/15  train CRPS 0.01171
fold 0 CRPS 0.01245 | floor 0.01427 | skill vs floor +12.78%


In [12]:
from sklearn.metrics import r2_score

print("floor per fold (recheck against Cell 5):", np.round(fl, 5))

scores, exp_all, y_all_te = [], [], []
for k in range(5):
    tr, te = folds[k]
    te = te[s_all[te] >= 2018]
    print(f"--- fold {k} ---")
    model = train_fold(tr)
    cdf = predict_cdf(model, te)
    scores.append(crps(cdf, y_all[te].astype(float)))
    pmf = np.diff(cdf, axis=1, prepend=0.0)          # probability of each exact yardage
    exp_all.append(pmf @ yards_grid)                 # expected yards for each play
    y_all_te.append(y_all[te])
    print(f"fold {k}: CRPS {scores[-1]:.5f}  floor {fl[k]:.5f}  skill {1 - scores[-1]/fl[k]:+.2%}  "
          f"R2 on Yards {r2_score(y_all[te], exp_all[-1]):.4f}")

print("\nmean CRPS:", round(np.mean(scores), 5), "| mean floor:", round(np.mean(fl), 5))
print(f"skill vs floor: {1 - np.mean(scores)/np.mean(fl):+.2%}   (hand-built: ~+2.4%, The Zoo: ~+15%)")
print("pooled R2 on Yards:", round(r2_score(np.concatenate(y_all_te), np.concatenate(exp_all)), 4),
      "  (hand-built model on the same 2018+ rows: ~0.03)")

floor per fold (recheck against Cell 5): [0.01427 0.01436 0.01447 0.01419 0.01406]
--- fold 0 ---
  epoch  1/15  train CRPS 0.03537
  epoch  6/15  train CRPS 0.01244
  epoch 11/15  train CRPS 0.01204
  epoch 15/15  train CRPS 0.01173
fold 0: CRPS 0.01245  floor 0.01427  skill +12.72%  R2 on Yards 0.1327
--- fold 1 ---
  epoch  1/15  train CRPS 0.03532
  epoch  6/15  train CRPS 0.01239
  epoch 11/15  train CRPS 0.01196
  epoch 15/15  train CRPS 0.01167
fold 1: CRPS 0.01260  floor 0.01436  skill +12.26%  R2 on Yards 0.1107
--- fold 2 ---
  epoch  1/15  train CRPS 0.03543
  epoch  6/15  train CRPS 0.01239
  epoch 11/15  train CRPS 0.01197
  epoch 15/15  train CRPS 0.01166
fold 2: CRPS 0.01263  floor 0.01447  skill +12.68%  R2 on Yards 0.1319
--- fold 3 ---
  epoch  1/15  train CRPS 0.03534
  epoch  6/15  train CRPS 0.01240
  epoch 11/15  train CRPS 0.01195
  epoch 15/15  train CRPS 0.01168
fold 3: CRPS 0.01238  floor 0.01419  skill +12.74%  R2 on Yards 0.1521
--- fold 4 ---
  epoch  1/15 